In [34]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

In [35]:
p = Path.cwd()
root = p.parents[2]
print(f'root:{root})')
data_dir = root.parent / 'data' / 'tiingo'
print(f'data folder:{data_dir}')

root:c:\Users\luciu\code\trading_research)
data folder:c:\Users\luciu\code\data\tiingo


In [36]:
panel = pd.read_parquet(f"{data_dir}/commods/commods_panel.parquet")
panel.columns.names = ["ticker", "field"]
panel.index.name = "date"

px_commods = panel.xs("adjClose",  axis=1, level="field")
vol_commods = panel.xs("adjVolume", axis=1, level="field")

In [37]:
px_commods

ticker,CANE,CORN,CPER,GLD,KRBN,SLV,SOYB,UGA,UNL,USL,WEAT
date,,,,,,,,,,,
2004-11-18,NaN,NaN,NaN,44.38,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2004-11-19,NaN,NaN,NaN,44.78,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2004-11-22,NaN,NaN,NaN,44.95,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2004-11-23,NaN,NaN,NaN,44.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2004-11-24,NaN,NaN,NaN,45.05,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2026-09-02,11.66,20.22,39.53,402.78,35.0900,59.07,27.62,128.91,5.9541,53.4100,27.86
2026-09-03,11.39,20.17,39.91,410.22,34.7437,60.55,27.79,131.21,5.8605,53.1790,27.12
2026-09-04,11.42,20.07,39.95,406.77,34.8800,59.82,27.65,133.22,5.8639,53.4800,26.49


In [38]:
panel = pd.read_parquet(f"{data_dir}/equity/equities_panel.parquet")
panel.columns.names = ["ticker", "field"]
panel.index.name = "date"

px_eq = panel.xs("adjClose",  axis=1, level="field")
vol_eq = panel.xs("adjVolume", axis=1, level="field")

In [39]:
px_eq

ticker,EEM,EFA,EWJ,IWM,QQQ,SPY
date,,,,,,
1993-01-29,NaN,NaN,NaN,NaN,NaN,24.100764
1993-02-01,NaN,NaN,NaN,NaN,NaN,24.272178
1993-02-02,NaN,NaN,NaN,NaN,NaN,24.323574
1993-02-03,NaN,NaN,NaN,NaN,NaN,24.580722
1993-02-04,NaN,NaN,NaN,NaN,NaN,24.683571
...,...,...,...,...,...,...
2026-09-02,67.15,107.06,96.04,294.01,709.24,765.160000
2026-09-03,67.47,108.21,97.90,295.19,717.67,773.170000
2026-09-04,68.70,108.35,98.28,296.01,718.96,770.190000


In [40]:
pxs = pd.concat([px_commods,px_eq], axis = 1, join = 'outer')

C:\Users\luciu\AppData\Local\Temp\ipykernel_32728\3618516015.py:1: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  pxs = pd.concat([px_commods,px_eq], axis = 1, join = 'outer')


In [41]:
pxs

ticker,CANE,CORN,CPER,GLD,KRBN,SLV,SOYB,UGA,UNL,USL,WEAT,EEM,EFA,EWJ,IWM,QQQ,SPY
date,,,,,,,,,,,,,,,,,
1993-01-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.100764
1993-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.272178
1993-02-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.323574
1993-02-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.580722
1993-02-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.683571
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-09-02,11.66,20.22,39.53,402.78,35.0900,59.07,27.62,128.91,5.9541,53.4100,27.86,67.15,107.06,96.04,294.01,709.24,765.160000
2026-09-03,11.39,20.17,39.91,410.22,34.7437,60.55,27.79,131.21,5.8605,53.1790,27.12,67.47,108.21,97.90,295.19,717.67,773.170000
2026-09-04,11.42,20.07,39.95,406.77,34.8800,59.82,27.65,133.22,5.8639,53.4800,26.49,68.70,108.35,98.28,296.01,718.96,770.190000


In [42]:
prices = pxs.sort_index()
max_ffill= 3
min_obs= 250

# drop instruments with too little history to estimate anything on
keep = prices.notna().sum() >= min_obs
prices = prices.loc[:, keep]

observed = prices.notna()

# never fill before an instrument exists
live = observed.cummax()

filled = prices.ffill(limit=max_ffill).where(live)
mask = filled.notna()

In [ ]:
filled = filled.drop(columns='KRBN')
first_print = filled.notna().idxmax().max()
log_filled = np.log(filled)

In [66]:
log_filled.max()

ticker
CANE    3.269949
CORN    3.964044
CPER    3.714791
GLD     6.206374
SLV     4.659658
SOYB    3.377929
UGA     4.912728
UNL     4.033531
USL     4.475631
WEAT    4.842359
EEM     4.265633
EFA     4.689603
EWJ     4.589752
IWM     5.720607
QQQ     6.613838
SPY     6.656572
dtype: float64

In [63]:
fast_ma = 63
slow_ma = 126

fast = log_filled.rolling(fast_ma).mean()
slow = log_filled.rolling(slow_ma).mean()

slow_std = log_filled.rolling(slow_ma).std()

trend = (fast-slow)/slow_std

In [67]:
trend.max()

ticker
CANE    0.940604
CORN    0.954221
CPER    0.953641
GLD     0.954651
SLV     0.927556
SOYB    0.927501
UGA     0.938814
UNL     0.903190
USL     0.947835
WEAT    0.955526
EEM     0.927216
EFA     0.926855
EWJ     0.949145
IWM     0.934220
QQQ     0.922577
SPY     0.919969
dtype: float64

In [68]:
trend.min()

ticker
CANE   -0.955631
CORN   -0.940698
CPER   -0.930722
GLD    -0.901063
SLV    -0.906802
SOYB   -0.946345
UGA    -0.935003
UNL    -0.945666
USL    -0.939647
WEAT   -0.934463
EEM    -0.929031
EFA    -0.940964
EWJ    -0.943550
IWM    -0.942175
QQQ    -0.943570
SPY    -0.951259
dtype: float64

In [ ]:
#sigmoid = x/1+exp(-x)
